# NDJF Manual Verification and Position QA

This notebook adds peak-position diagnostics, simple candidate intensity metrics, a manual-review scaffold, and batch quicklook plots for visual event verification.

In [ ]:
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/angelicasophyaramirez-blip/JPCZcatalogcolab.git"
BRANCH = "main"
REPO_DIR = "/content/JPCZcatalog"
FORCE_REFRESH_REPO = True
PERSIST_OUTPUTS_TO_DRIVE = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/JPCZcatalog_outputs"

if PERSIST_OUTPUTS_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    print("Persistent output dir:", DRIVE_OUTPUT_DIR)

os.chdir("/content")

if FORCE_REFRESH_REPO and os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print("Removed existing repo clone:", REPO_DIR)

if not os.path.exists(REPO_DIR):
    proc = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR],
        text=True,
        capture_output=True,
    )
    print(proc.stdout)
    print(proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{proc.stderr}")

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements-colab.txt"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR],
        check=True,
    )
else:
    print("Using existing repo clone:", REPO_DIR)

os.chdir(REPO_DIR)
src_dir = os.path.join(REPO_DIR, "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

print("Working directory:", os.getcwd())


In [ ]:
from pathlib import Path
import shutil

import pandas as pd

from jpcz_catalog.analysis import (
    add_candidate_intensity_metrics,
    add_japan_local_time_columns,
    add_position_group_columns,
    annotate_events_with_peak_position,
    build_manual_verification_scaffold,
)
from jpcz_catalog.era5 import open_arco_era5
from jpcz_catalog.radiation import AUTO_ERA5_OLR_TOKEN, resolve_era5_olr_variable
from jpcz_catalog.verification import render_manual_verification_summary, write_text_summary

CATALOG_PATH = Path(DRIVE_OUTPUT_DIR) / "jpcz_catalog_ndjf_merged_12h.csv"
AUTO_CATALOG_PATH = Path("outputs/verification/jpcz_catalog_ndjf_merged_12h_position_intensity.csv")
SCAFFOLD_PATH = Path("outputs/verification/jpcz_catalog_ndjf_merged_12h_manual_verification.csv")
SUMMARY_PATH = Path("outputs/verification/ndjf_manual_verification_merged_12h_summary.md")
CLOUD_VARIABLE = AUTO_ERA5_OLR_TOKEN
BASE_PLOT_DIR_NAME = "ndjf_event_quicklooks_merged_12h"
PLOT_DIR = Path("outputs/verification") / BASE_PLOT_DIR_NAME
SATELLITE_DIR = Path("outputs/verification/ndjf_event_satellite_panels_merged_12h")
OLR_DIR = Path("outputs/verification/ndjf_event_olr_panels_merged_12h")
SATELLITE_PANEL_LABEL = "MODIS daily true color (NASA GIBS)"
OLR_PANEL_LABEL = "ERA5-derived OLR cloud proxy"
PATTERN_TYPE_OPTIONS = [
    "",
    "clean band",
    "Hokkaido wake",
    "coastal Japan/Korea",
    "Sea of Japan circulation",
    "Pacific off Japan (N-S)",
    "Pacific off Japan (E-W)",
    "complex/other",
    "uncertain",
]

FORCE_REBUILD_AUGMENTED = False
FORCE_REBUILD_SCAFFOLD = False

drive_auto_catalog_path = Path(DRIVE_OUTPUT_DIR) / AUTO_CATALOG_PATH.name
drive_scaffold_path = Path(DRIVE_OUTPUT_DIR) / SCAFFOLD_PATH.name

REVIEW_TEXT_COLUMNS = [
    "verified_event",
    "cloud_band_present",
    "position_group_manual",
    "pattern_type_manual",
    "convergence_intensity_manual",
    "satellite_checked",
    "satellite_cloud_band_match",
    "satellite_source",
    "satellite_notes",
    "upper_level_forcing_note",
    "verification_notes",
]
REVIEW_NUMERIC_COLUMNS = [
    "manual_peak_convergence_lat",
    "manual_peak_convergence_lon",
]

def ensure_auto_columns(catalog_like_df: pd.DataFrame) -> pd.DataFrame:
    repaired = catalog_like_df.copy()

    candidate_columns = {
        "candidate_peak_convergence_1e5_s-1",
        "candidate_duration_weighted_convergence",
        "candidate_positive_box_vorticity_1e5_s-1",
        "candidate_peak_duration_vorticity_index",
        "candidate_peak_convergence_percentile",
        "convergence_intensity_auto",
    }
    if candidate_columns.difference(repaired.columns) and {
        "event_peak_D_1e5_s-1",
        "zeta_box_mean_s-1",
        "duration_hours",
    }.issubset(repaired.columns):
        repaired = add_candidate_intensity_metrics(repaired)

    position_columns = {"lat_position_group_auto", "lon_position_group_auto"}
    if position_columns.difference(repaired.columns) and {
        "peak_max_convergence_lat",
        "peak_max_convergence_lon",
    }.issubset(repaired.columns):
        repaired = add_position_group_columns(repaired)

    if {"event_peak_jst", "event_peak_jst_hour"}.difference(repaired.columns) and {
        "event_start",
        "event_end",
        "event_peak",
    }.issubset(repaired.columns):
        repaired = add_japan_local_time_columns(repaired)

    return repaired

def ensure_review_schema(catalog_like_df: pd.DataFrame) -> pd.DataFrame:
    repaired = ensure_auto_columns(catalog_like_df)
    repaired = build_manual_verification_scaffold(repaired)
    for column_name in REVIEW_TEXT_COLUMNS:
        repaired[column_name] = repaired[column_name].fillna("").astype("object")
    for column_name in REVIEW_NUMERIC_COLUMNS:
        repaired[column_name] = pd.to_numeric(repaired[column_name], errors="coerce")
    return repaired

def load_existing_csv(local_path: Path):
    drive_path = Path(DRIVE_OUTPUT_DIR) / local_path.name
    for candidate_path in (drive_path, local_path):
        if candidate_path.exists():
            return pd.read_csv(
                candidate_path,
                parse_dates=["event_start", "event_end", "event_peak"],
            ), candidate_path
    return None, None

AUTO_CATALOG_PATH.parent.mkdir(parents=True, exist_ok=True)
catalog_df = pd.read_csv(
    CATALOG_PATH,
    parse_dates=["event_start", "event_end", "event_peak"],
)

augmented_catalog, augmented_source = (None, None)
verification_scaffold, scaffold_source = (None, None)
ds = None

if not FORCE_REBUILD_AUGMENTED:
    augmented_catalog, augmented_source = load_existing_csv(AUTO_CATALOG_PATH)

if augmented_catalog is None:
    ds = open_arco_era5()
    if CLOUD_VARIABLE not in (None, AUTO_ERA5_OLR_TOKEN) and CLOUD_VARIABLE not in ds.data_vars:
        print(f"Requested cloud variable '{CLOUD_VARIABLE}' is not available in this ERA5 store. Falling back to wind + convergence only.")
        CLOUD_VARIABLE = None

    augmented_catalog = annotate_events_with_peak_position(ds, catalog_df)
    augmented_catalog = add_candidate_intensity_metrics(augmented_catalog)
    augmented_catalog = add_position_group_columns(augmented_catalog)
    augmented_source = Path("recomputed from merged catalog")
else:
    augmented_catalog = ensure_auto_columns(augmented_catalog)

if CLOUD_VARIABLE is not None and ds is None:
    ds = open_arco_era5()
    if CLOUD_VARIABLE not in (AUTO_ERA5_OLR_TOKEN,) and CLOUD_VARIABLE not in ds.data_vars:
        print(f"Requested cloud variable '{CLOUD_VARIABLE}' is not available in this ERA5 store. Falling back to wind + convergence only.")
        CLOUD_VARIABLE = None

augmented_catalog.to_csv(AUTO_CATALOG_PATH, index=False)
if PERSIST_OUTPUTS_TO_DRIVE:
    shutil.copy2(AUTO_CATALOG_PATH, drive_auto_catalog_path)

if not FORCE_REBUILD_SCAFFOLD:
    verification_scaffold, scaffold_source = load_existing_csv(SCAFFOLD_PATH)

if verification_scaffold is None:
    verification_scaffold = build_manual_verification_scaffold(augmented_catalog)
    scaffold_source = Path("rebuilt from augmented catalog")
else:
    verification_scaffold = ensure_review_schema(verification_scaffold)

verification_scaffold.to_csv(SCAFFOLD_PATH, index=False)
if PERSIST_OUTPUTS_TO_DRIVE:
    shutil.copy2(SCAFFOLD_PATH, drive_scaffold_path)

summary_text = render_manual_verification_summary(
    total_events=len(verification_scaffold),
    auto_catalog_name=AUTO_CATALOG_PATH.name,
    scaffold_name=SCAFFOLD_PATH.name,
    plot_dir_name=PLOT_DIR.name,
    cloud_variable=CLOUD_VARIABLE,
)
summary_path = write_text_summary(SUMMARY_PATH, summary_text)

if PERSIST_OUTPUTS_TO_DRIVE:
    shutil.copy2(summary_path, Path(DRIVE_OUTPUT_DIR) / summary_path.name)

print(summary_text)
print(f"Catalog source: {CATALOG_PATH}")
print(f"Augmented catalog source: {augmented_source or AUTO_CATALOG_PATH}")
print(f"Scaffold source: {scaffold_source or SCAFFOLD_PATH}")
verification_scaffold.head()


In [ ]:
if "ds" not in globals() or ds is None:
    ds = open_arco_era5()

from jpcz_catalog.radiation import AUTO_ERA5_OLR_TOKEN, available_era5_olr_variables, resolve_era5_olr_variable

candidate_variables = available_era5_olr_variables(ds)
resolved_variable = resolve_era5_olr_variable(ds, CLOUD_VARIABLE)

print(f"Total variables in ARCO ERA5 store: {len(ds.data_vars)}")
print("Candidate ERA5 OLR-like variables in the ARCO cloud store:")
for name in candidate_variables:
    print(" -", name)

if CLOUD_VARIABLE == AUTO_ERA5_OLR_TOKEN:
    if resolved_variable is None:
        print("\nAUTO_ERA5_OLR is requested, but no supported ERA5 OLR-like variable was found in this store.")
    else:
        print(f"\nAUTO_ERA5_OLR resolved to: {resolved_variable}")
elif CLOUD_VARIABLE is None:
    print("\nCLOUD_VARIABLE is currently None. No ERA5 OLR panel will be generated.")
else:
    print(f"\nCLOUD_VARIABLE is currently set to: {CLOUD_VARIABLE}")


In [ ]:
from pathlib import Path
import shutil

import numpy as np
import pandas as pd

from jpcz_catalog.analysis import load_event_peak_snapshot
from jpcz_catalog.config import JPCZ_POLYGON_VERTICES, VORTICITY_BOX, WORKING_DOMAIN
from jpcz_catalog.detect import compute_divergence_field, prepare_detection_geometry
from jpcz_catalog.era5 import open_arco_era5
from jpcz_catalog.plotting import plot_event_peak_quicklook, plot_scalar_field_map
from jpcz_catalog.radiation import AUTO_ERA5_OLR_TOKEN, era5_olr_like_field, resolve_era5_olr_variable
from jpcz_catalog.satellite import build_gibs_wms_getmap_url, default_modis_layers_for_date, download_gibs_image

PLOT_START = 0
PLOT_STOP = 12
OVERWRITE_EXISTING_PLOTS = False
CONTINUE_ON_PLOT_ERROR = True

if "verification_scaffold" not in globals():
    scaffold_input_path = SCAFFOLD_PATH if SCAFFOLD_PATH.exists() else Path(DRIVE_OUTPUT_DIR) / SCAFFOLD_PATH.name
    verification_scaffold = pd.read_csv(
        scaffold_input_path,
        parse_dates=["event_start", "event_end", "event_peak"],
    )
if "ensure_review_schema" in globals():
    verification_scaffold = ensure_review_schema(verification_scaffold)
if "ds" not in globals() or ds is None:
    ds = open_arco_era5()

resolved_olr_variable = resolve_era5_olr_variable(ds, CLOUD_VARIABLE)
if CLOUD_VARIABLE == AUTO_ERA5_OLR_TOKEN and resolved_olr_variable is None:
    print("No supported ERA5 OLR-like variable is available in this ARCO store. Satellite review will still work.")

PLOT_DIR.mkdir(parents=True, exist_ok=True)
drive_plot_dir = Path(DRIVE_OUTPUT_DIR) / PLOT_DIR.name
SATELLITE_DIR.mkdir(parents=True, exist_ok=True)
drive_satellite_dir = Path(DRIVE_OUTPUT_DIR) / SATELLITE_DIR.name
OLR_DIR.mkdir(parents=True, exist_ok=True)
drive_olr_dir = Path(DRIVE_OUTPUT_DIR) / OLR_DIR.name
if PERSIST_OUTPUTS_TO_DRIVE:
    drive_plot_dir.mkdir(parents=True, exist_ok=True)
    drive_satellite_dir.mkdir(parents=True, exist_ok=True)
    drive_olr_dir.mkdir(parents=True, exist_ok=True)

def quicklook_name_for_row(row_index: int, row: pd.Series) -> str:
    return f"{pd.Timestamp(row['event_peak']).strftime('%Y%m%dT%H%M')}_idx{row_index:03d}.png"

def satellite_name_for_row(row_index: int, row: pd.Series) -> str | None:
    satellite_layers = default_modis_layers_for_date(pd.Timestamp(row['event_peak']).normalize())
    if not satellite_layers:
        return None
    satellite_layer = satellite_layers[0]
    layer_slug = (
        satellite_layer.replace("MODIS_", "")
        .replace("_CorrectedReflectance_TrueColor", "")
        .lower()
    )
    return f"{pd.Timestamp(row['event_peak']).strftime('%Y%m%dT%H%M')}_idx{row_index:03d}_{layer_slug}.jpg"

def generate_review_panels(
    plot_start: int,
    plot_stop: int,
    *,
    overwrite_existing: bool = False,
    continue_on_error: bool = True,
):
    selected_events = verification_scaffold.iloc[plot_start:plot_stop].copy()
    geometry = None
    counts = {
        "rows": 0,
        "convergence_saved": 0,
        "olr_saved": 0,
        "satellite_saved": 0,
        "errors": 0,
    }

    for idx, row in selected_events.iterrows():
        counts["rows"] += 1
        try:
            out_name = quicklook_name_for_row(idx, row)
            out_path = PLOT_DIR / out_name
            drive_out_path = drive_plot_dir / out_name
            olr_out_path = OLR_DIR / out_name
            drive_olr_out_path = drive_olr_dir / out_name

            peak_snapshot = load_event_peak_snapshot(
                ds,
                row["event_peak"],
                domain=WORKING_DOMAIN,
                cloud_variable=resolved_olr_variable,
            )

            if geometry is None:
                geometry = prepare_detection_geometry(
                    peak_snapshot.longitude,
                    peak_snapshot.latitude,
                    JPCZ_POLYGON_VERTICES,
                )

            divergence_field = compute_divergence_field(
                peak_snapshot,
                dx=geometry.dx,
                dy=geometry.dy,
            )

            olr_field = None
            olr_label = OLR_PANEL_LABEL
            if resolved_olr_variable is not None:
                olr_field, olr_label = era5_olr_like_field(peak_snapshot, resolved_olr_variable)

            event_peak_utc = pd.Timestamp(row["event_peak"])
            event_peak_jst = pd.Timestamp(row.get("event_peak_jst", event_peak_utc + pd.Timedelta(hours=9)))

            if not (out_path.exists() and not overwrite_existing):
                if drive_out_path.exists() and not overwrite_existing:
                    shutil.copy2(drive_out_path, out_path)
                    print(f"Copied existing Drive convergence plot: {out_name}")
                else:
                    plot_event_peak_quicklook(
                        peak_snapshot,
                        divergence_field,
                        domain=WORKING_DOMAIN,
                        polygon_vertices=JPCZ_POLYGON_VERTICES,
                        vorticity_box=VORTICITY_BOX,
                        title=(
                            f"NDJF event {idx} | peak {event_peak_utc.strftime('%Y-%m-%d %H:%M UTC')} | {event_peak_jst.strftime('%Y-%m-%d %H:%M JST')}\n"
                            f"peak D={row['event_peak_D_1e5_s-1']:.2f} | duration={int(row['duration_hours'])} h | auto lat group={row['lat_position_group_auto']}"
                        ),
                        cloud_field=None,
                        max_location=(row["peak_max_convergence_lat"], row["peak_max_convergence_lon"]),
                        centroid_location=(row["peak_convergence_centroid_lat"], row["peak_convergence_centroid_lon"]),
                        save_path=out_path,
                    )
                    if PERSIST_OUTPUTS_TO_DRIVE:
                        shutil.copy2(out_path, drive_out_path)
                    counts["convergence_saved"] += 1
                    print(f"Saved convergence plot: {out_path}")

            if olr_field is not None and not (olr_out_path.exists() and not overwrite_existing):
                if drive_olr_out_path.exists() and not overwrite_existing:
                    shutil.copy2(drive_olr_out_path, olr_out_path)
                    print(f"Copied existing Drive OLR panel: {out_name}")
                else:
                    plot_scalar_field_map(
                        olr_field,
                        domain=WORKING_DOMAIN,
                        title=(
                            f"NDJF event {idx} | ERA5-derived OLR cloud proxy\n"
                            f"peak {event_peak_utc.strftime('%Y-%m-%d %H:%M UTC')} | {event_peak_jst.strftime('%Y-%m-%d %H:%M JST')}"
                        ),
                        colorbar_label=olr_label,
                        polygon_vertices=JPCZ_POLYGON_VERTICES,
                        boxes={"Shinoda vorticity box": VORTICITY_BOX},
                        levels=np.arange(140, 331, 10),
                        cmap="Greys_r",
                        extend="both",
                        save_path=olr_out_path,
                    )
                    if PERSIST_OUTPUTS_TO_DRIVE:
                        shutil.copy2(olr_out_path, drive_olr_out_path)
                    counts["olr_saved"] += 1
                    print(f"Saved OLR panel: {olr_out_path}")

            satellite_layers = default_modis_layers_for_date(event_peak_utc.normalize())
            if satellite_layers:
                satellite_layer = satellite_layers[0]
                layer_slug = (
                    satellite_layer.replace("MODIS_", "")
                    .replace("_CorrectedReflectance_TrueColor", "")
                    .lower()
                )
                satellite_name = f"{pd.Timestamp(row['event_peak']).strftime('%Y%m%dT%H%M')}_idx{idx:03d}_{layer_slug}.jpg"
                satellite_out_path = SATELLITE_DIR / satellite_name
                drive_satellite_out_path = drive_satellite_dir / satellite_name

                if not (satellite_out_path.exists() and not overwrite_existing):
                    if drive_satellite_out_path.exists() and not overwrite_existing:
                        shutil.copy2(drive_satellite_out_path, satellite_out_path)
                        print(f"Copied existing Drive satellite panel: {satellite_name}")
                    else:
                        satellite_url = build_gibs_wms_getmap_url(
                            layer_name=satellite_layer,
                            target_date=event_peak_utc.normalize(),
                            bbox=WORKING_DOMAIN,
                            width=1400,
                            height=1000,
                            image_format="image/jpeg",
                        )
                        try:
                            download_gibs_image(satellite_url, satellite_out_path)
                            if PERSIST_OUTPUTS_TO_DRIVE:
                                shutil.copy2(satellite_out_path, drive_satellite_out_path)
                            counts["satellite_saved"] += 1
                            print(f"Saved satellite panel: {satellite_out_path}")
                        except Exception as exc:
                            print(f"Satellite download failed for row {idx}: {exc}")
            else:
                print(f"No MODIS true-color satellite panel available for {event_peak_utc:%Y-%m-%d}.")
        except Exception as exc:
            counts["errors"] += 1
            print(f"Panel generation failed for row {idx}: {exc}")
            if not continue_on_error:
                raise

    return counts

counts = generate_review_panels(
    PLOT_START,
    PLOT_STOP,
    overwrite_existing=OVERWRITE_EXISTING_PLOTS,
    continue_on_error=CONTINUE_ON_PLOT_ERROR,
)
print("Generation summary:", counts)


In [ ]:
from pathlib import Path

import pandas as pd

if "verification_scaffold" not in globals():
    scaffold_input_path = SCAFFOLD_PATH if SCAFFOLD_PATH.exists() else Path(DRIVE_OUTPUT_DIR) / SCAFFOLD_PATH.name
    verification_scaffold = pd.read_csv(
        scaffold_input_path,
        parse_dates=["event_start", "event_end", "event_peak"],
    )
if "ensure_review_schema" in globals():
    verification_scaffold = ensure_review_schema(verification_scaffold)

preview_columns = [
    "event_peak",
    "event_peak_D_1e5_s-1",
    "event_peak_jst",
    "event_peak_jst_hour",
    "duration_hours",
    "peak_max_convergence_lat",
    "peak_max_convergence_lon",
    "peak_convergence_centroid_lat",
    "peak_convergence_centroid_lon",
    "lat_position_group_auto",
    "lon_position_group_auto",
    "candidate_peak_convergence_1e5_s-1",
    "candidate_duration_weighted_convergence",
    "candidate_positive_box_vorticity_1e5_s-1",
    "candidate_peak_duration_vorticity_index",
    "candidate_peak_convergence_percentile",
    "convergence_intensity_auto",
    "verified_event",
    "cloud_band_present",
    "position_group_manual",
    "pattern_type_manual",
    "convergence_intensity_manual",
    "satellite_checked",
    "satellite_cloud_band_match",
    "satellite_source",
]

display(verification_scaffold.reindex(columns=preview_columns).iloc[PLOT_START:PLOT_STOP])


In [ ]:
from pathlib import Path

import pandas as pd
from jpcz_catalog.satellite import default_modis_layers_for_date

if "verification_scaffold" not in globals():
    scaffold_input_path = SCAFFOLD_PATH if SCAFFOLD_PATH.exists() else Path(DRIVE_OUTPUT_DIR) / SCAFFOLD_PATH.name
    verification_scaffold = pd.read_csv(
        scaffold_input_path,
        parse_dates=["event_start", "event_end", "event_peak"],
    )
if "ensure_review_schema" in globals():
    verification_scaffold = ensure_review_schema(verification_scaffold)

BULK_BUILD_START = 0
BULK_BUILD_STOP = len(verification_scaffold)
BATCH_SIZE = 25
BUILD_ONLY_MISSING = True
SHOW_MISSING_SUMMARY = False
RUN_BULK_BUILD = False
BULK_OVERWRITE_EXISTING = False

local_plot_dir = PLOT_DIR
drive_plot_dir = Path(DRIVE_OUTPUT_DIR) / PLOT_DIR.name
local_satellite_dir = SATELLITE_DIR
drive_satellite_dir = Path(DRIVE_OUTPUT_DIR) / SATELLITE_DIR.name
local_olr_dir = OLR_DIR
drive_olr_dir = Path(DRIVE_OUTPUT_DIR) / OLR_DIR.name

def expected_quicklook_name(row_index: int, row: pd.Series) -> str:
    return f"{pd.Timestamp(row['event_peak']).strftime('%Y%m%dT%H%M')}_idx{row_index:03d}.png"

def expected_satellite_name(row_index: int, row: pd.Series) -> str | None:
    layers = default_modis_layers_for_date(pd.Timestamp(row['event_peak']).normalize())
    if not layers:
        return None
    layer_slug = (
        layers[0].replace("MODIS_", "")
        .replace("_CorrectedReflectance_TrueColor", "")
        .lower()
    )
    return f"{pd.Timestamp(row['event_peak']).strftime('%Y%m%dT%H%M')}_idx{row_index:03d}_{layer_slug}.jpg"

def missing_flags(row_index: int, row: pd.Series) -> dict:
    base_name = expected_quicklook_name(row_index, row)
    convergence_missing = not ((local_plot_dir / base_name).exists() or (drive_plot_dir / base_name).exists())
    olr_missing = not ((local_olr_dir / base_name).exists() or (drive_olr_dir / base_name).exists())
    satellite_name = expected_satellite_name(row_index, row)
    if satellite_name is None:
        satellite_missing = False
    else:
        satellite_missing = not ((local_satellite_dir / satellite_name).exists() or (drive_satellite_dir / satellite_name).exists())
    return {
        "convergence_missing": convergence_missing,
        "olr_missing": olr_missing,
        "satellite_missing": satellite_missing,
    }

subset = verification_scaffold.iloc[BULK_BUILD_START:BULK_BUILD_STOP]
missing_rows = []
missing_counts = {
    "convergence": 0,
    "olr": 0,
    "satellite": 0,
}

for idx, row in subset.iterrows():
    flags = missing_flags(idx, row)
    if flags["convergence_missing"]:
        missing_counts["convergence"] += 1
    if flags["olr_missing"]:
        missing_counts["olr"] += 1
    if flags["satellite_missing"]:
        missing_counts["satellite"] += 1
    if (not BUILD_ONLY_MISSING) or any(flags.values()):
        missing_rows.append(idx)

batch_ranges = []
for start in range(0, len(missing_rows), BATCH_SIZE):
    batch = missing_rows[start:start + BATCH_SIZE]
    if batch:
        batch_ranges.append((batch[0], batch[-1] + 1))

if RUN_BULK_BUILD:
    if "generate_review_panels" not in globals():
        raise RuntimeError("Run cell 4 first so generate_review_panels() is available.")
    print(f"Rows in requested bulk window: {len(subset)}")
    print(f"Missing convergence plots: {missing_counts['convergence']}")
    print(f"Missing OLR panels: {missing_counts['olr']}")
    print(f"Missing satellite panels (where MODIS should exist): {missing_counts['satellite']}")
    print(f"Rows queued for generation: {len(missing_rows)}")
    print("Suggested cell 4 batch ranges:")
    for start, stop in batch_ranges[:20]:
        print(f" - PLOT_START = {start}, PLOT_STOP = {stop}")
    if len(batch_ranges) > 20:
        print(f" ... and {len(batch_ranges) - 20} more batches")

    all_counts = []
    for start, stop in batch_ranges:
        print(f"\nRunning batch: {start} to {stop}")
        batch_counts = generate_review_panels(
            start,
            stop,
            overwrite_existing=BULK_OVERWRITE_EXISTING,
            continue_on_error=True,
        )
        all_counts.append((start, stop, batch_counts))
        print(f"Batch summary: {batch_counts}")

    print("\nBulk generation complete.")
    for start, stop, batch_counts in all_counts:
        print(f" - {start}:{stop} -> {batch_counts}")
elif SHOW_MISSING_SUMMARY:
    print(f"Rows in requested bulk window: {len(subset)}")
    print(f"Missing convergence plots: {missing_counts['convergence']}")
    print(f"Missing OLR panels: {missing_counts['olr']}")
    print(f"Missing satellite panels (where MODIS should exist): {missing_counts['satellite']}")
    print(f"Rows queued for generation: {len(missing_rows)}")
    print("Suggested cell 4 batch ranges:")
    for start, stop in batch_ranges[:20]:
        print(f" - PLOT_START = {start}, PLOT_STOP = {stop}")
    if len(batch_ranges) > 20:
        print(f" ... and {len(batch_ranges) - 20} more batches")
else:
    print("Bulk helper idle. Leave RUN_BULK_BUILD = False during normal review sessions.")


In [ ]:
from pathlib import Path

import pandas as pd

if "verification_scaffold" not in globals():
    scaffold_input_path = SCAFFOLD_PATH if SCAFFOLD_PATH.exists() else Path(DRIVE_OUTPUT_DIR) / SCAFFOLD_PATH.name
    verification_scaffold = pd.read_csv(
        scaffold_input_path,
        parse_dates=["event_start", "event_end", "event_peak"],
    )
if "ensure_review_schema" in globals():
    verification_scaffold = ensure_review_schema(verification_scaffold)

RUN_EVENT_SEARCH = False
SEARCH_START_UTC = "2018-02-03 00:00"
SEARCH_END_UTC = "2018-02-04 23:59"
SEARCH_MATCH_MODE = "overlap"  # overlap, peak_only, start_only

if RUN_EVENT_SEARCH:
    search_start = pd.Timestamp(SEARCH_START_UTC)
    search_end = pd.Timestamp(SEARCH_END_UTC)

    if SEARCH_MATCH_MODE == "peak_only":
        search_mask = verification_scaffold["event_peak"].between(search_start, search_end)
    elif SEARCH_MATCH_MODE == "start_only":
        search_mask = verification_scaffold["event_start"].between(search_start, search_end)
    else:
        search_mask = (
            (verification_scaffold["event_start"] <= search_end)
            & (verification_scaffold["event_end"] >= search_start)
        )

    match_columns = [
        "event_start",
        "event_end",
        "event_peak",
        "event_peak_jst",
        "duration_hours",
        "event_peak_D_1e5_s-1",
        "lat_position_group_auto",
        "verified_event",
        "pattern_type_manual",
    ]
    matched_events = verification_scaffold.loc[search_mask].reindex(columns=match_columns)

    print(f"Search window UTC: {search_start} to {search_end}")
    print(f"Match mode: {SEARCH_MATCH_MODE}")
    print(f"Matched events: {len(matched_events)}")

    display(matched_events)

    if not matched_events.empty:
        first_idx = int(matched_events.index[0])
        last_idx = int(matched_events.index[-1]) + 1
        print(f"Suggested cell 4 range: PLOT_START = {first_idx}, PLOT_STOP = {last_idx}")
        if "current_index" in globals():
            current_index.value = first_idx
            if "jump_row_widget" in globals():
                jump_row_widget.value = first_idx
            print(f"Widget jumped to row {first_idx}.")
    else:
        print("No catalog events matched that window.")
else:
    print("Event search helper idle. Use the date search controls in the widget for normal review.")


In [ ]:
from pathlib import Path

import pandas as pd
import ipywidgets as widgets
from IPython.display import HTML, Image, clear_output, display
from jpcz_catalog.satellite import default_modis_layers_for_date

if "verification_scaffold" not in globals():
    scaffold_input_path = SCAFFOLD_PATH if SCAFFOLD_PATH.exists() else Path(DRIVE_OUTPUT_DIR) / SCAFFOLD_PATH.name
    verification_scaffold = pd.read_csv(
        scaffold_input_path,
        parse_dates=["event_start", "event_end", "event_peak"],
    )
if "ensure_review_schema" in globals():
    verification_scaffold = ensure_review_schema(verification_scaffold)

local_plot_dir = PLOT_DIR
drive_plot_dir = Path(DRIVE_OUTPUT_DIR) / PLOT_DIR.name
local_satellite_dir = SATELLITE_DIR
drive_satellite_dir = Path(DRIVE_OUTPUT_DIR) / SATELLITE_DIR.name
local_olr_dir = OLR_DIR
drive_olr_dir = Path(DRIVE_OUTPUT_DIR) / OLR_DIR.name
drive_scaffold_path = Path(DRIVE_OUTPUT_DIR) / SCAFFOLD_PATH.name

def quicklook_filename(row_index: int, row: pd.Series) -> str:
    timestamp = pd.Timestamp(row['event_peak']).strftime('%Y%m%dT%H%M')
    return f"{timestamp}_idx{row_index:03d}.png"

def quicklook_path(row_index: int, row: pd.Series) -> Path:
    name = quicklook_filename(row_index, row)
    local_path = local_plot_dir / name
    if local_path.exists():
        return local_path
    return drive_plot_dir / name

def satellite_filename(row_index: int, row: pd.Series):
    satellite_layers = default_modis_layers_for_date(pd.Timestamp(row['event_peak']).normalize())
    if not satellite_layers:
        return None
    satellite_layer = satellite_layers[0]
    layer_slug = (
        satellite_layer.replace("MODIS_", "")
        .replace("_CorrectedReflectance_TrueColor", "")
        .lower()
    )
    timestamp = pd.Timestamp(row['event_peak']).strftime('%Y%m%dT%H%M')
    return f"{timestamp}_idx{row_index:03d}_{layer_slug}.jpg"

def satellite_path(row_index: int, row: pd.Series):
    name = satellite_filename(row_index, row)
    if name is None:
        return None
    local_path = local_satellite_dir / name
    if local_path.exists():
        return local_path
    return drive_satellite_dir / name

def olr_path(row_index: int, row: pd.Series) -> Path:
    name = quicklook_filename(row_index, row)
    local_path = local_olr_dir / name
    if local_path.exists():
        return local_path
    return drive_olr_dir / name

def event_option_label(row_index: int) -> str:
    row = verification_scaffold.iloc[row_index]
    start = pd.Timestamp(row['event_start']).strftime('%Y-%m-%d %H:%M')
    end = pd.Timestamp(row['event_end']).strftime('%Y-%m-%d %H:%M')
    peak = pd.Timestamp(row['event_peak']).strftime('%Y-%m-%d %H:%M')
    return f"{row_index} | {start} to {end} | peak {peak}"

def matched_indices_for_query(query: str) -> list[int]:
    query = query.strip()
    if not query:
        return []

    timestamp = pd.Timestamp(query)
    if len(query) <= 10:
        search_start = timestamp.normalize()
        search_end = search_start + pd.Timedelta(days=1) - pd.Timedelta(minutes=1)
    else:
        search_start = timestamp
        search_end = timestamp

    overlap_mask = (
        (verification_scaffold['event_start'] <= search_end)
        & (verification_scaffold['event_end'] >= search_start)
    )
    peak_mask = verification_scaffold['event_peak'].between(search_start, search_end)
    matches = verification_scaffold.index[overlap_mask | peak_mask]
    return [int(idx) for idx in matches]

current_index = widgets.BoundedIntText(value=0, min=0, max=len(verification_scaffold) - 1, description='Row')
jump_row_widget = widgets.BoundedIntText(value=0, min=0, max=len(verification_scaffold) - 1, description='Jump row')
date_query_widget = widgets.Text(description='Find date', placeholder='YYYY-MM-DD or YYYY-MM-DD HH:MM')
match_select_widget = widgets.Dropdown(options=[], description='Matches')
find_date_button = widgets.Button(description='Find date')
go_match_button = widgets.Button(description='Go match')
jump_row_button = widgets.Button(description='Go row')
verified_widget = widgets.Dropdown(
    options=['', 'yes', 'no', 'uncertain'],
    description='Verified',
)
cloud_widget = widgets.Dropdown(
    options=['', 'yes', 'no', 'uncertain'],
    description='Cloud proxy',
)
position_widget = widgets.Dropdown(
    options=['', 'north-shifted', 'central', 'south-shifted', 'west-shifted', 'east-shifted', 'complex'],
    description='Manual pos',
)
pattern_widget = widgets.Dropdown(
    options=PATTERN_TYPE_OPTIONS,
    description='Pattern',
)
convergence_intensity_widget = widgets.Dropdown(
    options=['', 'low-convergence', 'moderate-convergence', 'high-convergence', 'uncertain'],
    description='Conv int',
)
manual_lat_widget = widgets.Text(description='Manual lat')
manual_lon_widget = widgets.Text(description='Manual lon')
satellite_checked_widget = widgets.Dropdown(
    options=['', 'planned', 'yes', 'no'],
    description='Satellite',
)
satellite_match_widget = widgets.Dropdown(
    options=['', 'yes', 'no', 'uncertain'],
    description='Sat match',
)
satellite_source_widget = widgets.Text(description='Sat source')
forcing_widget = widgets.Text(description='Jet note')
notes_widget = widgets.Textarea(description='Notes', layout=widgets.Layout(width='100%', height='120px'))
satellite_notes_widget = widgets.Textarea(description='Sat notes', layout=widgets.Layout(width='100%', height='90px'))
save_button = widgets.Button(description='Save row', button_style='success')
next_button = widgets.Button(description='Save + next', button_style='primary')
prev_button = widgets.Button(description='Prev row')
jump_unreviewed_button = widgets.Button(description='Next blank')
status_html = widgets.HTML()
meta_html = widgets.HTML()
image_out = widgets.Output()

def normalize_float(value):
    if value is None or value == '':
        return pd.NA
    try:
        if pd.isna(value):
            return pd.NA
    except TypeError:
        pass
    return float(value)

def save_current(_=None):
    idx = int(current_index.value)
    verification_scaffold.at[idx, 'verified_event'] = verified_widget.value
    verification_scaffold.at[idx, 'cloud_band_present'] = cloud_widget.value
    verification_scaffold.at[idx, 'position_group_manual'] = position_widget.value
    verification_scaffold.at[idx, 'pattern_type_manual'] = pattern_widget.value
    verification_scaffold.at[idx, 'convergence_intensity_manual'] = convergence_intensity_widget.value
    verification_scaffold.at[idx, 'manual_peak_convergence_lat'] = normalize_float(manual_lat_widget.value)
    verification_scaffold.at[idx, 'manual_peak_convergence_lon'] = normalize_float(manual_lon_widget.value)
    verification_scaffold.at[idx, 'satellite_checked'] = satellite_checked_widget.value
    verification_scaffold.at[idx, 'satellite_cloud_band_match'] = satellite_match_widget.value
    verification_scaffold.at[idx, 'satellite_source'] = satellite_source_widget.value
    verification_scaffold.at[idx, 'satellite_notes'] = satellite_notes_widget.value
    verification_scaffold.at[idx, 'upper_level_forcing_note'] = forcing_widget.value
    verification_scaffold.at[idx, 'verification_notes'] = notes_widget.value
    verification_scaffold.to_csv(SCAFFOLD_PATH, index=False)
    if PERSIST_OUTPUTS_TO_DRIVE:
        verification_scaffold.to_csv(drive_scaffold_path, index=False)
    status_html.value = f"<b>Saved row {idx}</b> to {SCAFFOLD_PATH.name}"

def load_row(idx: int):
    row = verification_scaffold.iloc[idx]
    jump_row_widget.value = idx
    verified_widget.value = row.get('verified_event', '') if pd.notna(row.get('verified_event', '')) else ''
    cloud_widget.value = row.get('cloud_band_present', '') if pd.notna(row.get('cloud_band_present', '')) else ''
    position_widget.value = row.get('position_group_manual', '') if pd.notna(row.get('position_group_manual', '')) else ''
    pattern_widget.value = row.get('pattern_type_manual', '') if pd.notna(row.get('pattern_type_manual', '')) else ''
    convergence_intensity_widget.value = row.get('convergence_intensity_manual', '') if pd.notna(row.get('convergence_intensity_manual', '')) else ''
    manual_lat_widget.value = str(float(row['manual_peak_convergence_lat'])) if pd.notna(row.get('manual_peak_convergence_lat')) else ''
    manual_lon_widget.value = str(float(row['manual_peak_convergence_lon'])) if pd.notna(row.get('manual_peak_convergence_lon')) else ''
    satellite_checked_widget.value = row.get('satellite_checked', '') if pd.notna(row.get('satellite_checked', '')) else ''
    satellite_match_widget.value = row.get('satellite_cloud_band_match', '') if pd.notna(row.get('satellite_cloud_band_match', '')) else ''
    satellite_source_widget.value = row.get('satellite_source', '') if pd.notna(row.get('satellite_source', '')) else ''
    satellite_notes_widget.value = row.get('satellite_notes', '') if pd.notna(row.get('satellite_notes', '')) else ''
    forcing_widget.value = row.get('upper_level_forcing_note', '') if pd.notna(row.get('upper_level_forcing_note', '')) else ''
    notes_widget.value = row.get('verification_notes', '') if pd.notna(row.get('verification_notes', '')) else ''

    plot_path = quicklook_path(idx, row)
    sat_path = satellite_path(idx, row)
    olr_plot_path = olr_path(idx, row)
    event_peak_utc = pd.Timestamp(row['event_peak'])
    event_peak_jst = pd.Timestamp(row.get('event_peak_jst', event_peak_utc + pd.Timedelta(hours=9)))
    meta_html.value = (
        f"<b>event_peak UTC:</b> {event_peak_utc} &nbsp; &nbsp; "
        f"<b>event_peak JST:</b> {event_peak_jst} &nbsp; &nbsp; "
        f"<b>JST hour:</b> {row.get('event_peak_jst_hour', event_peak_jst.hour)}<br>"
        f"<b>month:</b> {row['month']} &nbsp; &nbsp; "
        f"<b>peak D:</b> {row['event_peak_D_1e5_s-1']:.2f} &nbsp; &nbsp; "
        f"<b>duration:</b> {int(row['duration_hours'])} h<br>"
        f"<b>auto max lat/lon:</b> {row['peak_max_convergence_lat']:.2f}, {row['peak_max_convergence_lon']:.2f} &nbsp; &nbsp; "
        f"<b>auto centroid lat/lon:</b> {row['peak_convergence_centroid_lat']:.2f}, {row['peak_convergence_centroid_lon']:.2f}<br>"
        f"<b>auto lat group:</b> {row['lat_position_group_auto']} &nbsp; &nbsp; "
        f"<b>auto lon group:</b> {row['lon_position_group_auto']} &nbsp; &nbsp; "
        f"<b>auto conv intensity:</b> {row.get('convergence_intensity_auto', '')}<br>"
        f"<b>peak-convergence percentile:</b> {row.get('candidate_peak_convergence_percentile', float('nan')):.1f} &nbsp; &nbsp; "
        f"<b>cloud proxy:</b> {OLR_PANEL_LABEL} &nbsp; &nbsp; "
        f"<b>pattern:</b> {row.get('pattern_type_manual', '')}"
    )

    with image_out:
        clear_output(wait=True)
        if plot_path.exists():
            display(HTML("<b>Convergence plot</b>"))
            display(Image(filename=str(plot_path), width=1350))
        else:
            display(HTML(f"<b>Missing quicklook:</b> {plot_path}<br>Generate cell 4 for the row range that includes {idx}."))

        if sat_path is None:
            display(HTML("<b>Satellite panel unavailable.</b> No MODIS true-color image is available for this event date."))
        elif sat_path.exists():
            display(HTML(f"<b>Satellite panel</b> ({SATELLITE_PANEL_LABEL})"))
            display(Image(filename=str(sat_path), width=1350))
        else:
            display(HTML(f"<b>Missing satellite panel:</b> {sat_path}<br>Generate cell 4 for the row range that includes {idx}."))

        if olr_plot_path.exists():
            display(HTML(f"<b>OLR cloud-proxy panel</b> ({OLR_PANEL_LABEL})"))
            display(Image(filename=str(olr_plot_path), width=1350))
        elif CLOUD_VARIABLE is None:
            display(HTML("<b>No ERA5 OLR panel configured.</b> Set the OLR cloud proxy in cells 2-3 if you want that fallback panel."))
        else:
            display(HTML(f"<b>Missing OLR panel:</b> {olr_plot_path}<br>Generate cell 4 for the row range that includes {idx}."))

def on_row_change(change):
    if change['name'] == 'value':
        load_row(int(change['new']))

def go_to_row(_=None):
    current_index.value = int(jump_row_widget.value)

def save_and_next(_):
    save_current()
    if current_index.value < current_index.max:
        current_index.value += 1

def go_prev(_):
    if current_index.value > current_index.min:
        current_index.value -= 1

def jump_next_blank(_):
    blanks = verification_scaffold.index[verification_scaffold['verified_event'].fillna('') == '']
    blanks = [i for i in blanks if i > current_index.value]
    if blanks:
        current_index.value = blanks[0]
        jump_row_widget.value = blanks[0]
    else:
        status_html.value = '<b>No later blank rows found.</b>'

def find_by_date(_=None):
    query = date_query_widget.value.strip()
    if not query:
        status_html.value = '<b>Enter a date or timestamp first.</b>'
        match_select_widget.options = []
        return

    try:
        matches = matched_indices_for_query(query)
    except Exception as exc:
        status_html.value = f'<b>Could not parse date query:</b> {exc}'
        match_select_widget.options = []
        return

    if not matches:
        status_html.value = f'<b>No catalog event matched:</b> {query}'
        match_select_widget.options = []
        return

    match_select_widget.options = [(event_option_label(idx), idx) for idx in matches]
    match_select_widget.value = matches[0]
    if len(matches) == 1:
        current_index.value = matches[0]
        jump_row_widget.value = matches[0]
        status_html.value = f'<b>Found 1 matching event.</b> Jumped to row {matches[0]}.'
    else:
        status_html.value = f'<b>Found {len(matches)} matching events.</b> Select one from Matches and click Go match.'

def go_to_match(_=None):
    if match_select_widget.value is None:
        status_html.value = '<b>No matched event selected.</b>'
        return
    current_index.value = int(match_select_widget.value)
    jump_row_widget.value = int(match_select_widget.value)
    status_html.value = f'<b>Jumped to row {int(match_select_widget.value)}.</b>'

current_index.observe(on_row_change)
save_button.on_click(save_current)
next_button.on_click(save_and_next)
prev_button.on_click(go_prev)
jump_row_button.on_click(go_to_row)
find_date_button.on_click(find_by_date)
go_match_button.on_click(go_to_match)
jump_unreviewed_button.on_click(jump_next_blank)

controls = widgets.VBox([
    widgets.HBox([current_index, jump_row_widget, jump_row_button, prev_button, save_button, next_button, jump_unreviewed_button]),
    widgets.HBox([date_query_widget, find_date_button, match_select_widget, go_match_button]),
    status_html,
    meta_html,
    widgets.HBox([verified_widget, cloud_widget, position_widget, pattern_widget]),
    convergence_intensity_widget,
    widgets.HBox([manual_lat_widget, manual_lon_widget]),
    widgets.HBox([satellite_checked_widget, satellite_match_widget]),
    satellite_source_widget,
    forcing_widget,
    notes_widget,
    satellite_notes_widget,
])

display(controls)
display(image_out)
jump_row_widget.value = int(current_index.value)
load_row(int(current_index.value))
